# Assignment 7 — Solving the Water-Jugs Problem Using Adversarial Search

This notebook implements the Water-Jugs problem as a **two-player adversarial game** and solves it using:

- **Minimax**
- **Alpha-Beta Pruning**

It also includes:

- a playable **human vs AI** loop,
- a **benchmark** to compare execution time,
- and a short **report** explaining the design choices.

---

## Problem idea

Two jugs have capacities **A** and **B** liters.  
A target volume **T** is given. Two players alternate moves.

A player wins immediately if, after their move, **either jug contains exactly `T` liters**.

If no valid move remains, the game ends in a **draw**.

---

## State representation

A state is represented as:

\[
(x, y)
\]

where:

- `x` = current volume in jug 1
- `y` = current volume in jug 2

The legal moves are:

1. Fill jug 1
2. Fill jug 2
3. Empty jug 1
4. Empty jug 2
5. Pour jug 1 → jug 2
6. Pour jug 2 → jug 1

---

## Important implementation note

A raw minimax search on this problem can revisit the same states and loop forever.  
To keep the search safe and practical, this notebook uses:

- **depth-limited search**
- **path-based cycle detection**

That makes the code usable for assignment/demo purposes while still showing the real minimax and alpha-beta structure.

## Utility function design

The assignment asks for:

- **positive scores** for states closer to the target for the AI,
- **negative scores** for states closer to the target for the opponent,
- **zero** for states with no valid moves left.

In practice, the notebook uses:

- **large terminal scores** for exact wins/losses,
- a **heuristic score** for non-terminal leaf nodes.

### Terminal states
- If the AI has just made `T` appear in either jug, score is **high positive**
- If the human has just made `T` appear, score is **high negative**
- If no moves remain, score is **0**

### Non-terminal leaf states
A simple distance-based heuristic is used:

\[
score = 10 - \min(|x - T|,\ |y - T|)
\]

Then the sign is adjusted from the AI perspective depending on whose turn it is.

This is not a perfect mathematical evaluation of future winning probability.  
It is a **reasonable heuristic** for assignment-level adversarial search.

In [5]:
from time import perf_counter

def valid_moves(state, capacities):
    """
    Return all legal next states from the current state.

    Parameters
    ----------
    state : tuple[int, int]
        Current water volumes (x, y).
    capacities : tuple[int, int]
        Jug capacities (A, B).

    Returns
    -------
    list[tuple[str, tuple[int, int]]]
        A list of (move_name, next_state).
    """
    x, y = state
    A, B = capacities

    moves = []
    seen = set()

    def add_move(name, next_state):
        if next_state != state and next_state not in seen:
            seen.add(next_state)
            moves.append((name, next_state))

    # Fill operations
    add_move(f"Fill jug 1 to {A}", (A, y))
    add_move(f"Fill jug 2 to {B}", (x, B))

    # Empty operations
    add_move("Empty jug 1", (0, y))
    add_move("Empty jug 2", (x, 0))

    # Pour jug 1 -> jug 2
    pour = min(x, B - y)
    add_move("Pour jug 1 -> jug 2", (x - pour, y + pour))

    # Pour jug 2 -> jug 1
    pour = min(y, A - x)
    add_move("Pour jug 2 -> jug 1", (x + pour, y - pour))

    return moves


def is_goal(state, target):
    """Check whether either jug contains exactly the target volume."""
    x, y = state
    return x == target or y == target


def terminal_utility(state, is_ai_turn, target):
    """
    Return a terminal utility if the state is a win/loss.
    Returns None if the state is not terminal.
    """
    if is_goal(state, target):
        # If the current state is terminal, the previous player made the winning move.
        previous_player_was_ai = not is_ai_turn
        return 1000 if previous_player_was_ai else -1000
    return None


def heuristic_utility(state, is_ai_turn, target):
    """
    Heuristic score used at depth cutoffs.

    Positive -> favorable for AI
    Negative -> favorable for human/opponent
    """
    x, y = state
    distance = min(abs(x - target), abs(y - target))
    base_score = max(0, 10 - distance)

    return base_score if is_ai_turn else -base_score

In [6]:
def minimax(state, depth, is_ai_turn, capacities, target, path=None):
    """
    Depth-limited minimax with cycle detection.

    Returns
    -------
    (score, best_move)
    """
    if path is None:
        path = set()

    terminal_score = terminal_utility(state, is_ai_turn, target)
    if terminal_score is not None:
        return terminal_score, None

    moves = valid_moves(state, capacities)
    if depth == 0 or not moves:
        return heuristic_utility(state, is_ai_turn, target) if moves else 0, None

    key = (state, is_ai_turn)
    if key in path:
        # Cycle detected: treat as a neutral draw-like state.
        return 0, None

    path.add(key)
    try:
        best_move = None

        if is_ai_turn:
            best_score = float("-inf")
            for move_name, next_state in moves:
                score, _ = minimax(next_state, depth - 1, False, capacities, target, path)
                if score > best_score:
                    best_score = score
                    best_move = (move_name, next_state)
            return best_score, best_move

        else:
            best_score = float("inf")
            for move_name, next_state in moves:
                score, _ = minimax(next_state, depth - 1, True, capacities, target, path)
                if score < best_score:
                    best_score = score
                    best_move = (move_name, next_state)
            return best_score, best_move
    finally:
        path.remove(key)


def alphabeta(state, depth, is_ai_turn, capacities, target, alpha=float("-inf"), beta=float("inf"), path=None):
    """
    Depth-limited minimax with alpha-beta pruning.
    """
    if path is None:
        path = set()

    terminal_score = terminal_utility(state, is_ai_turn, target)
    if terminal_score is not None:
        return terminal_score, None

    moves = valid_moves(state, capacities)
    if depth == 0 or not moves:
        return heuristic_utility(state, is_ai_turn, target) if moves else 0, None

    key = (state, is_ai_turn)
    if key in path:
        return 0, None

    path.add(key)
    try:
        best_move = None

        if is_ai_turn:
            value = float("-inf")
            for move_name, next_state in moves:
                score, _ = alphabeta(next_state, depth - 1, False, capacities, target, alpha, beta, path)
                if score > value:
                    value = score
                    best_move = (move_name, next_state)
                alpha = max(alpha, value)
                if alpha >= beta:
                    break
            return value, best_move

        else:
            value = float("inf")
            for move_name, next_state in moves:
                score, _ = alphabeta(next_state, depth - 1, True, capacities, target, alpha, beta, path)
                if score < value:
                    value = score
                    best_move = (move_name, next_state)
                beta = min(beta, value)
                if alpha >= beta:
                    break
            return value, best_move
    finally:
        path.remove(key)

In [7]:
def format_state(state):
    return f"({state[0]}, {state[1]})"


def print_moves(moves):
    for i, (name, next_state) in enumerate(moves, start=1):
        print(f"{i}. {name:22s} -> {format_state(next_state)}")


def get_human_move(state, capacities, target):
    moves = valid_moves(state, capacities)
    if not moves:
        return None

    print("\nAvailable moves:")
    print_moves(moves)

    while True:
        choice = input("Choose a move number: ").strip()
        if not choice.isdigit():
            print("Enter a valid integer choice.")
            continue

        idx = int(choice)
        if 1 <= idx <= len(moves):
            return moves[idx - 1]
        print("Choice out of range.")


def play_game(capacities=(4, 7), target=2, ai_starts=False, depth=8, use_alpha_beta=True):
    """
    Play a human vs AI game from the notebook.

    The human is jug-player 1 by convention in the prompt flow,
    but both players use the same state transition rules.
    """
    state = (0, 0)
    is_ai_turn = ai_starts

    print(f"\nCapacities: {capacities}")
    print(f"Target: {target}")
    print("Enter moves by number when it is your turn.")
    print("--------------------------------------------------")

    while True:
        if is_goal(state, target):
            winner = "AI" if not is_ai_turn else "Human"
            print(f"\nGame over. {winner} wins with state {format_state(state)}.")
            return winner

        moves = valid_moves(state, capacities)
        if not moves:
            print(f"\nGame over. No valid moves remain. Draw at state {format_state(state)}.")
            return "Draw"

        print(f"\nCurrent state: {format_state(state)} | Turn: {'AI' if is_ai_turn else 'Human'}")

        if is_ai_turn:
            if use_alpha_beta:
                score, chosen = alphabeta(state, depth, True, capacities, target)
            else:
                score, chosen = minimax(state, depth, True, capacities, target)

            if chosen is None:
                print("AI has no chosen move. Declaring draw.")
                return "Draw"

            move_name, next_state = chosen
            print(f"AI chooses: {move_name} -> {format_state(next_state)} | score = {score}")
            state = next_state
            is_ai_turn = False
        else:
            chosen = get_human_move(state, capacities, target)
            if chosen is None:
                print("No valid human move. Declaring draw.")
                return "Draw"

            move_name, next_state = chosen
            state = next_state
            is_ai_turn = True

In [8]:
def benchmark(capacities=(4, 7), target=2, start_state=(0, 0), depths=(6, 7, 8, 9, 10)):
    """
    Compare execution time of plain minimax vs alpha-beta pruning.

    Returns a list of dicts for easy printing.
    """
    results = []

    for depth in depths:
        t0 = perf_counter()
        score_mm, move_mm = minimax(start_state, depth, True, capacities, target)
        t1 = perf_counter()

        score_ab, move_ab = alphabeta(start_state, depth, True, capacities, target)
        t2 = perf_counter()

        results.append({
            "depth": depth,
            "minimax_time_sec": t1 - t0,
            "alphabeta_time_sec": t2 - t1,
            "minimax_score": score_mm,
            "alphabeta_score": score_ab,
            "minimax_move": move_mm[0] if move_mm else None,
            "alphabeta_move": move_ab[0] if move_ab else None,
        })

    return results


def show_benchmark_table(results):
    headers = [
        "depth", "minimax_time_sec", "alphabeta_time_sec",
        "minimax_score", "alphabeta_score", "minimax_move", "alphabeta_move"
    ]
    print("\t".join(headers))
    for row in results:
        print(
            f"{row['depth']}\t"
            f"{row['minimax_time_sec']:.6f}\t"
            f"{row['alphabeta_time_sec']:.6f}\t"
            f"{row['minimax_score']}\t"
            f"{row['alphabeta_score']}\t"
            f"{row['minimax_move']}\t"
            f"{row['alphabeta_move']}"
        )

In [ ]:
play_game(capacities=(4, 7), target=2, ai_starts=False, depth=8, use_alpha_beta=True)


Capacities: (4, 7)
Target: 2
Enter moves by number when it is your turn.
--------------------------------------------------

Current state: (0, 0) | Turn: Human

Available moves:
1. Fill jug 1 to 4        -> (4, 0)
2. Fill jug 2 to 7        -> (0, 7)
Choice out of range.

Current state: (0, 7) | Turn: AI
AI chooses: Fill jug 1 to 4 -> (4, 7) | score = 0

Current state: (4, 7) | Turn: Human

Available moves:
1. Empty jug 1            -> (0, 7)
2. Empty jug 2            -> (4, 0)
Choice out of range.
Choice out of range.
Choice out of range.

Current state: (0, 7) | Turn: AI
AI chooses: Fill jug 1 to 4 -> (4, 7) | score = 0

Current state: (4, 7) | Turn: Human

Available moves:
1. Empty jug 1            -> (0, 7)
2. Empty jug 2            -> (4, 0)
Enter a valid integer choice.
Enter a valid integer choice.
Enter a valid integer choice.
Enter a valid integer choice.
Enter a valid integer choice.


## How to run

### 1) Compare search speed
Run:

```python
results = benchmark()
show_benchmark_table(results)
```

### 2) Start a human vs AI game
Run:

```python
play_game(capacities=(4, 7), target=2, ai_starts=False, depth=8, use_alpha_beta=True)
```

You can change:

- `capacities`
- `target`
- `depth`
- `ai_starts`
- `use_alpha_beta`

---

## What to expect

- **Alpha-beta pruning** should usually explore fewer nodes than plain minimax.
- For small games, the speedup may be modest.
- For larger capacities or deeper search, pruning becomes much more valuable.

## Brief report

### 1. Rules and structure of the game
The game is a deterministic, turn-based adversarial game.  
Each move changes the current state `(x, y)` using one of the six legal operations.  
The game ends immediately when either jug contains exactly `T` liters.

### 2. Design choices for the utility function
The utility function gives:
- strong positive reward for AI wins,
- strong negative reward for human wins,
- zero for dead-end states,
- and a heuristic score based on distance to the target at depth cutoffs.

This makes the search practical without pretending the state evaluation is exact.

### 3. Results of testing the AI
The benchmark cell compares runtime for:
- plain minimax,
- alpha-beta pruning.

Expected behavior:
- both algorithms should produce the **same move choice** at the same search depth,
- alpha-beta should usually be **faster** because it prunes branches that cannot affect the final decision.

### 4. Limitations
- The search is **depth-limited**, so it is not a complete proof of optimal play for arbitrarily long games.
- The heuristic is simple and may miss deeper tactical traps.
- Path-based cycle detection is necessary because the water-jug state graph contains loops.